<a href="https://colab.research.google.com/github/oktennant-del/CUNY---DC37--Data-Analytics-Academy/blob/main/Python_Project_3_April_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NYPD Arrest Datat Analytics - Python Project 3

This project analyzes NYPD arrest data from NYC Open Data.
The goal is to compare arrest patterns between two ZIP codes and explore trends using descriptive statistics, data cleaning, and visualization.

Dataset source: https://opendata.cityofnewyork.us/

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


In [11]:
df = pd.read_excel('NYPD_Arrest_Data_(Year_to_Date)_20260411.csv.xlsx')
df.head()

,ARREST_KEY,ARREST_DATE,PD_CD,PD_DESC,KY_CD,OFNS_DESC,LAW_CODE,LAW_CAT_CD,ARREST_BORO,ARREST_PRECINCT,JURISDICTION_CODE,AGE_GROUP,PERP_SEX,PERP_RACE,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,Location
0,299201470,2025-01-10,105,STRANGULATION 1ST,106.0,FELONY ASSAULT,PL 1211200,F,Q,110,0,25-44,F,WHITE,1017105,210917,40.745543,-73.881427,POINT (-73.881427 40.745543)
1,299351927,2025-01-13,105,STRANGULATION 1ST,106.0,FELONY ASSAULT,PL 1211200,F,S,120,0,25-44,M,BLACK,962808,174275,40.645005,-74.077265,POINT (-74.077265 40.645005)
2,299366743,2025-01-13,157,RAPE 1,104.0,RAPE,PL 130351A,F,B,46,0,25-44,M,BLACK HISPANIC,1011755,250279,40.853598,-73.900577,POINT (-73.9005768807295 40.8535983673823)
3,300990163,2025-02-12,153,RAPE 3,104.0,RAPE,PL 1302504,F,K,77,0,(null),(null),BLACK,1003509,185018,40.674496,-73.930571,POINT (-73.9305713255961 40.6744956865259)
4,301010124,2025-02-12,157,RAPE 1,104.0,RAPE,PL 130353A,F,K,77,0,25-44,M,BLACK,1003509,185018,40.674496,-73.930571,POINT (-73.9305713255961 40.6744956865259)


In [13]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df = df.drop_duplicates()

df['arrest_precinct'] = df['arrest_precinct'].fillna("Unknown")
df['arrest_boro'] = df['arrest_boro'].fillna("Unknown")
# The column 'arrest_zip_code' does not exist in the DataFrame.
# Please verify the correct column name for zip codes in your dataset.
# df['arrest_zip_code'] = df['arrest_zip_code'].fillna("00000")

df.head()

,arrest_key,arrest_date,pd_cd,pd_desc,ky_cd,ofns_desc,law_code,law_cat_cd,arrest_boro,arrest_precinct,jurisdiction_code,age_group,perp_sex,perp_race,x_coord_cd,y_coord_cd,latitude,longitude,location
0,299201470,2025-01-10,105,STRANGULATION 1ST,106.0,FELONY ASSAULT,PL 1211200,F,Q,110,0,25-44,F,WHITE,1017105,210917,40.745543,-73.881427,POINT (-73.881427 40.745543)
1,299351927,2025-01-13,105,STRANGULATION 1ST,106.0,FELONY ASSAULT,PL 1211200,F,S,120,0,25-44,M,BLACK,962808,174275,40.645005,-74.077265,POINT (-74.077265 40.645005)
2,299366743,2025-01-13,157,RAPE 1,104.0,RAPE,PL 130351A,F,B,46,0,25-44,M,BLACK HISPANIC,1011755,250279,40.853598,-73.900577,POINT (-73.9005768807295 40.8535983673823)
3,300990163,2025-02-12,153,RAPE 3,104.0,RAPE,PL 1302504,F,K,77,0,(null),(null),BLACK,1003509,185018,40.674496,-73.930571,POINT (-73.9305713255961 40.6744956865259)
4,301010124,2025-02-12,157,RAPE 1,104.0,RAPE,PL 130353A,F,K,77,0,25-44,M,BLACK,1003509,185018,40.674496,-73.930571,POINT (-73.9305713255961 40.6744956865259)


In [23]:
df.columns

Index(['arrest_key', 'arrest_date', 'pd_cd', 'pd_desc', 'ky_cd', 'ofns_desc',
       'law_code', 'law_cat_cd', 'arrest_boro', 'arrest_precinct',
       'jurisdiction_code', 'age_group', 'perp_sex', 'perp_race', 'x_coord_cd',
       'y_coord_cd', 'latitude', 'longitude', 'location'],
      dtype='object')

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="geoapi_colab_project")
geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_zip(lat, lon):
    try:
        loc = geocode(f"{lat}, {lon}", exactly_one=True)
        if loc and "address" in loc.raw and "postcode" in loc.raw["address"]:
            return loc.raw["address"]["postcode"]
        return None
    except:
        return None

# Create the column
df["arrest_zip_code"] = df.apply(lambda row: get_zip(row["latitude"], row["longitude"]), axis=1)


In [ ]:
zip1 = "10027"
zip2 = "11207"

df_zip1 = df[df["arrest_zip_code"] == zip1]
df_zip2 = df[df["arrest_zip_code"] == zip2]

len(df_zip1), len(df_zip2)


We will compare:
- ZIP Code **10027** (Harlem)
- ZIP Code **11207** (East New York)

Feature selected: **Age Group of arrestees**


In [ ]:
desc_stats = pd.DataFrame({
    "ZIP": [zip1, zip2],
    "Total Arrests": [len(df_zip1), len(df_zip2)],
    "Most Common Age Group": [
        df_zip1['age_group'].mode()[0],
        df_zip2['age_group'].mode()[0]
    ],
    "Most Common Offense": [
        df_zip1['ofns_desc'].mode()[0],
        df_zip2['ofns_desc'].mode()[0]
    ]
})

desc_stats


In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=df[df['arrest_zip_code'].isin([zip1, zip2])],
              x='age_group', hue='arrest_zip_code')

plt.title("Age Group Distribution of Arrests: 10027 vs 11207")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.legend(title="ZIP Code")
plt.xticks(rotation=45)
plt.show()


In [ ]:
df['arrest_date'] = pd.to_datetime(df['arrest_date'])

df_zip = df[df['arrest_zip_code'].isin([zip1, zip2])]
df_zip['month'] = df_zip['arrest_date'].dt.to_period('M')

monthly = df_zip.groupby(['month', 'arrest_zip_code']).size().reset_index(name='count')

plt.figure(figsize=(12,6))
sns.lineplot(data=monthly, x='month', y='count', hue='arrest_zip_code', marker='o')
plt.title("Monthly Arrest Trends: ZIP 10027 vs 11207")
plt.xlabel("Month")
plt.ylabel("Arrest Count")
plt.xticks(rotation=45)
plt.show()


### Conclusion

ZIP Code 10027 and 11207 show different arrest patterns, with 11207 having a higher total number of arrests. Age group distributions also differ, with each ZIP showing distinct dominant age categories. The most common offenses vary between the two neighborhoods, suggesting different policing or crime patterns.

### Bias Discussion

Arrest data reflects policing activity, not necessarily actual crime rates. Neighborhoods with heavier police presence may show higher arrest counts regardless of underlying behavior. Demographic and socioeconomic factors can influence both policing strategies and arrest outcomes. Therefore, comparisons should be interpreted cautiously to avoid reinforcing systemic bias.
